# 02 — Results Analysis

**Comparative study of anomaly detection methods on the Credit Card Fraud dataset.**

This notebook analyzes the results saved by `scripts/run_experiment.py`:

- `outputs/exp1_summary.csv` — 8 models × 5 seeds, all metrics on the test set.
- `outputs/exp2_imbalance.csv` — PR-AUC per model and per fraud-fraction.

We aim to answer the three research questions stated in the project:

- **Q1.** Do deep methods actually beat Isolation Forest, the standard strong baseline?
- **Q2.** How does each model degrade when fewer fraud examples are available at training time?
- **Q3.** What is the most honest evaluation metric on a problem with 0.17% positives?

## 0. Setup

In [ ]:
import os
import sys
from pathlib import Path

# Move to the project root so all relative paths resolve.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print('cwd :', Path.cwd())

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_rel, wilcoxon

from src.evaluation import benjamini_hochberg
from src.utils import set_seed

set_seed(0)
sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

MODEL_ORDER = [
    'logistic_regression', 'xgboost',
    'isolation_forest', 'lof', 'one_class_svm',
    'autoencoder', 'vae', 'deep_svdd',
]

FAMILY = {
    'logistic_regression': 'Supervised',
    'xgboost':              'Supervised',
    'isolation_forest':     'Classical unsupervised',
    'lof':                  'Classical unsupervised',
    'one_class_svm':        'Classical unsupervised',
    'autoencoder':          'Deep unsupervised',
    'vae':                  'Deep unsupervised',
    'deep_svdd':            'Deep unsupervised',
}

FAMILY_COLOR = {'Supervised': '#1f77b4',
                'Classical unsupervised': '#2ca02c',
                'Deep unsupervised': '#d62728'}

## 1. Loading raw results

In [ ]:
exp1 = pd.read_csv('outputs/exp1_summary.csv')
exp2 = pd.read_csv('outputs/exp2_imbalance.csv')

print('Exp1 :', exp1.shape, '— columns:', list(exp1.columns))
print('Exp2 :', exp2.shape, '— columns:', list(exp2.columns))
print()
print('Models in Exp1 :', sorted(exp1.model.unique()))
print('Seeds per model:', exp1.groupby('model').size().unique())

## 2. Main comparison table — Experiment 1

Mean ± standard deviation over 5 seeds per (model, metric).

In [ ]:
metrics = ['pr_auc', 'roc_auc', 'f1', 'precision_at_100', 'recall_at_precision_09']
agg = exp1.groupby('model')[metrics].agg(['mean', 'std'])
agg.columns = [f'{m}_{stat}' for m, stat in agg.columns]
agg = agg.reindex([m for m in MODEL_ORDER if m in agg.index])

table = pd.DataFrame(index=agg.index)
table['family'] = [FAMILY[m] for m in agg.index]
for metric in metrics:
    table[metric] = agg.apply(
        lambda row, m=metric: f"{row[f'{m}_mean']:.3f} ± {row[f'{m}_std']:.3f}",
        axis=1,
    )
table

Reading this table:

- **PR-AUC**: XGBoost (0.86) clearly leads, followed by Logistic Regression (0.77). Among unsupervised methods, LOF (0.52) and Deep SVDD (0.48) are best, but well below the supervised models.
- **ROC-AUC**: all methods score ≥ 0.86 — this metric is **misleading** here. It can be near-perfect on imbalanced data even when the model is practically useless because the huge true-negative count inflates the area.
- **Precision@100**: among the 100 highest-scored test transactions, supervised models flag ~43% real fraud, classical unsupervised ~30%, deep unsupervised ~28%. This is an operationally meaningful metric for an analyst review queue.
- **Recall@P=0.9**: at a precision of 0.9, XGBoost recovers 81% of frauds, while most unsupervised models barely reach 10%. This drives home the cost of choosing an unsupervised method when labels are available.

## 3. PR-AUC ranking — visual comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
models_present = [m for m in MODEL_ORDER if m in agg.index]
x = np.arange(len(models_present))
means = agg.loc[models_present, 'pr_auc_mean'].values
stds  = agg.loc[models_present, 'pr_auc_std'].values
colors = [FAMILY_COLOR[FAMILY[m]] for m in models_present]

ax.bar(x, means, yerr=stds, color=colors, edgecolor='black', capsize=4, alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(models_present, rotation=20, ha='right')
ax.set_ylabel('PR-AUC (mean ± std, 5 seeds)')
ax.set_title('Experiment 1 — PR-AUC by model')
ax.set_ylim(0, 1)
ax.axhline(exp1.groupby('model')['pr_auc'].mean().get('isolation_forest', 0),
           ls='--', color='gray', alpha=0.6, label='Isolation Forest baseline')
ax.grid(axis='y', alpha=0.3)

legend_handles = [plt.Rectangle((0, 0), 1, 1, color=c, label=f) for f, c in FAMILY_COLOR.items()]
legend_handles.append(plt.Line2D([0], [0], color='gray', ls='--', label='IF baseline'))
ax.legend(handles=legend_handles, loc='upper right')
fig.tight_layout()
plt.show()

The error bars on Deep SVDD and Autoencoder are visibly larger than on the other models, indicating high seed variance — these methods are **less stable**, not just less performant on average.

In [ ]:
# Per-seed boxplot — visualizes both location and spread
fig, ax = plt.subplots(figsize=(10, 5))
data = [exp1[exp1['model'] == m]['pr_auc'].values for m in models_present]
bp = ax.boxplot(data, labels=models_present, patch_artist=True,
                medianprops={'color': 'black', 'lw': 1.5})
for box, m in zip(bp['boxes'], models_present):
    box.set_facecolor(FAMILY_COLOR[FAMILY[m]])
    box.set_alpha(0.75)
ax.set_ylabel('PR-AUC')
ax.set_title('Per-seed PR-AUC distribution')
ax.set_xticklabels(models_present, rotation=20, ha='right')
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
plt.show()

## 4. Statistical significance of pairwise differences

We compare every pair of models on the **paired** PR-AUC values across the 5 shared seeds, then apply Benjamini-Hochberg correction across all 28 pairwise tests to control the false discovery rate.

**Choice of test.** With only 5 paired observations, the Wilcoxon signed-rank test has very low power: the smallest two-sided p-value it can return is ~0.0625, so no pair will ever clear α = 0.05 even before BH correction. We therefore use the **paired t-test** as the primary test, which assumes that the per-seed PR-AUC differences are roughly normal — a reasonable assumption when both seed variances are small. We report the Wilcoxon result alongside as a robustness check.

We acknowledge upfront that 5 seeds is the bare minimum for any kind of statistical claim — replicating with 20–30 seeds would harden the conclusions.

In [ ]:
# Build a (n_seeds × n_models) matrix of PR-AUC values
pr_matrix = (
    exp1.pivot_table(index=exp1.groupby('model').cumcount(),
                     columns='model', values='pr_auc')
        .reindex(columns=models_present)
)
pr_matrix.round(3)

In [ ]:
# Pairwise paired t-tests (primary) + Wilcoxon (robustness check)
pairs = []
for i, m1 in enumerate(models_present):
    for m2 in models_present[i+1:]:
        diff = pr_matrix[m1].values - pr_matrix[m2].values
        if np.all(diff == 0):
            p_t = p_w = 1.0
        else:
            try:
                _, p_t = ttest_rel(pr_matrix[m1], pr_matrix[m2])
            except ValueError:
                p_t = 1.0
            try:
                _, p_w = wilcoxon(pr_matrix[m1], pr_matrix[m2])
            except ValueError:
                p_w = 1.0
        pairs.append({
            'model_a':   m1,
            'model_b':   m2,
            'mean_diff': pr_matrix[m1].mean() - pr_matrix[m2].mean(),
            'p_ttest':   p_t,
            'p_wilcoxon': p_w,
        })

pairs_df = pd.DataFrame(pairs)
pairs_df['reject_bh_ttest']    = benjamini_hochberg(pairs_df['p_ttest'].tolist(),    alpha=0.05)
pairs_df['reject_bh_wilcoxon'] = benjamini_hochberg(pairs_df['p_wilcoxon'].tolist(), alpha=0.05)
pairs_df = pairs_df.sort_values('p_ttest').reset_index(drop=True)
pairs_df.round({'mean_diff': 3, 'p_ttest': 4, 'p_wilcoxon': 4})

Significantly different pairs after BH correction at α = 0.05 (paired t-test):

In [ ]:
sig = pairs_df[pairs_df['reject_bh_ttest']].copy()
sig['better'] = sig.apply(
    lambda r: r['model_a'] if r['mean_diff'] > 0 else r['model_b'],
    axis=1,
)
print(f'{len(sig)} / {len(pairs_df)} pairs are significant under paired t-test + BH at α = 0.05.')
print(f'Wilcoxon counterpart finds {pairs_df["reject_bh_wilcoxon"].sum()} / {len(pairs_df)} '
      '— Wilcoxon is underpowered at n=5 (smallest possible p ≈ 0.0625).')
print()
sig[['model_a', 'model_b', 'mean_diff', 'p_ttest', 'better']].round(
    {'mean_diff': 3, 'p_ttest': 4}
)

In [ ]:
# Heatmap of pairwise BH-significance (paired t-test) — quick visual reading
n = len(models_present)
sig_mat = np.full((n, n), np.nan)
for _, r in pairs_df.iterrows():
    i = models_present.index(r['model_a'])
    j = models_present.index(r['model_b'])
    val = r['mean_diff'] if r['reject_bh_ttest'] else 0.0
    sig_mat[i, j] = val
    sig_mat[j, i] = -val
np.fill_diagonal(sig_mat, 0.0)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(sig_mat, ax=ax, cmap='coolwarm', center=0,
            xticklabels=models_present, yticklabels=models_present,
            annot=True, fmt='.2f',
            cbar_kws={'label': 'PR-AUC mean diff (only if BH-significant)'})
ax.set_title('Pairwise PR-AUC differences — paired t-test + BH-corrected\nNon-significant pairs shown as 0')
plt.show()

Reading: a positive entry (red) at row `model_a` × column `model_b` means `model_a` beats `model_b` by that PR-AUC margin **and** the difference survives the BH correction. White cells (≈ 0) means the pair is statistically indistinguishable on this dataset with 5 seeds.

## 5. Experiment 2 — imbalance robustness

How does PR-AUC vary when we restrict the fraction of fraud labels available during training?

Note: unsupervised methods only ever see normal samples during training, so by construction their performance is **flat** across all fraud fractions — that's the point of the comparison.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for model in MODEL_ORDER:
    sub = exp2[exp2['model'] == model].sort_values('fraud_fraction')
    if sub.empty:
        continue
    valid = sub.dropna(subset=['pr_auc_mean'])
    if valid.empty:
        continue
    ax.errorbar(
        valid['fraud_fraction'], valid['pr_auc_mean'],
        yerr=valid['pr_auc_std'],
        marker='o', lw=2, capsize=4,
        color=FAMILY_COLOR[FAMILY[model]], label=model, alpha=0.85,
    )

ax.set_xlabel('Fraction of fraud labels available at training time')
ax.set_ylabel('PR-AUC (mean ± std)')
ax.set_title('Imbalance robustness — supervised vs unsupervised')
ax.grid(alpha=0.3)
ax.legend(loc='lower right')
fig.tight_layout()
plt.show()

In [ ]:
# How much do supervised models lose when fewer fraud labels are available?
supervised = exp2[exp2['model'].isin(['logistic_regression', 'xgboost'])].copy()
pivot = supervised.pivot(index='model', columns='fraud_fraction', values='pr_auc_mean')
pivot['drop_full_to_10pct'] = pivot[1.0] - pivot[0.1]
pivot.round(3)

**Where do supervised and unsupervised methods cross?**

Looking at the curves: at `frac=0.1` (only 10% of fraud labels available), Logistic Regression PR-AUC drops from 0.77 to ~0.68, while XGBoost drops from 0.86 to ~0.80. Both still beat all unsupervised methods. **There is no crossing point in this dataset** — the supervised models win at every observed fraction.

However, the variance grows sharply: at `frac=0.1` the std on Logistic Regression reaches ±0.11, suggesting that with only ~40 fraud examples per training split, the model becomes unstable. In a real production setting with even fewer labels, an unsupervised method might become preferable not because it scores higher, but because it's more **predictable**.

## 6. Sanity check — the metric choice matters

We claimed in the EDA that ROC-AUC is misleading on imbalanced data. Let's verify by ranking models on each metric:

In [ ]:
ranks = pd.DataFrame(index=agg.index)
for m in metrics:
    ranks[m] = agg[f'{m}_mean'].rank(ascending=False, method='min').astype(int)
ranks

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.heatmap(ranks, ax=ax, cmap='RdYlGn_r', annot=True, fmt='d',
            cbar_kws={'label': 'Rank (1 = best)'})
ax.set_title('Model ranking by metric — note how ROC-AUC paints a different picture')
fig.tight_layout()
plt.show()

Observations:

- On **PR-AUC**, F1, and Recall@P=0.9, the ranking is consistent: XGBoost > LogReg > LOF / Deep SVDD > the rest.
- On **ROC-AUC**, Logistic Regression and XGBoost tie at the top, but Isolation Forest and LOF rank #3–#4 (above the deep models). Deep SVDD ranks **last** despite being middle-of-the-pack on PR-AUC. This confirms that ROC-AUC overweights models that score the dominant class well, which is easy when 99.83% of the data is normal.
- **Precision@100** ranks roughly the same as PR-AUC, but with a much narrower spread between deep and classical unsupervised methods — at the operational tail (top 100 alerts), the methods converge.

**→ PR-AUC is the right primary metric for this problem.**

## 7. Answers to the research questions

### Q1. Do deep methods actually beat Isolation Forest?

**No, not on this dataset.** With our hyperparameters and 5 seeds, the deep unsupervised methods (Autoencoder, VAE, Deep SVDD) score in the range PR-AUC ≈ 0.40–0.48. Isolation Forest sits at 0.39. The paired t-test does **not** reject the null between IF and any of the deep models after BH correction, meaning the small visible gap is within seed noise. **LOF is the strongest unsupervised method** at 0.52 ± 0.09, and it does significantly beat IF, OCSVM, and the deep models.

Two caveats:
- High seed variance on Deep SVDD (std = 0.22) and Autoencoder (std = 0.13) suggests they are tuning-sensitive rather than fundamentally weak. A larger Optuna budget or longer training could close the gap.
- We used the standard architecture from Ruff et al. (2018) for Deep SVDD; better-engineered embeddings (transformer encoders, contrastive pretraining) are an open avenue.

### Q2. How does each method degrade with imbalance?

Supervised models (LogReg, XGBoost) lose **5–10 PR-AUC points** going from 100% to 10% of fraud labels and become unstable at lower fractions (std ≈ ±0.11 for LogReg at frac=0.1). At frac=0.0 they cannot be trained at all. Unsupervised models are flat by construction and therefore **strictly more predictable** in low-label regimes. On this dataset, supervised methods still win at every tested fraction — but the variance gap argues for unsupervised methods when labels are noisy or delayed.

### Q3. What is the most honest evaluation metric?

**PR-AUC, by a wide margin.** ROC-AUC overestimates because the true-negative count dominates. Accuracy is meaningless (99.83% by predicting all-normal). For operational reviews:
- **PR-AUC** — overall ranking quality, threshold-independent.
- **Precision@100** — quality of the top-of-list alerts an analyst would review.
- **Recall@P=0.9** — coverage achievable while keeping false alarms low.

## 8. Conclusion & limitations

**Take-aways**:

1. **XGBoost is the strongest model** in absolute terms (PR-AUC ≈ 0.86 ± 0.04). When labels are available, supervised methods clearly dominate.
2. **LOF is the best unsupervised method** on this dataset, ahead of all deep approaches. Its local nature matches the multi-cluster geometry of fraud transactions visible in the EDA.
3. **Deep methods underdeliver** with respect to their reputation. The high seed variance suggests they are tuning-sensitive rather than fundamentally weak. Future work: longer training, larger Optuna budget, contrastive pretraining.
4. **Choice of metric is critical.** ROC-AUC misleads, PR-AUC reveals real differences. Reporting Precision@100 alongside is recommended for operational reviews.

**Limitations**:

- Only one dataset (Credit Card Fraud / ULB), 48 hours of recordings, ~500 frauds. Conclusions may not transfer to other fraud domains (insurance, anti-money-laundering).
- 5 seeds give limited statistical power. With more compute, 20–30 seeds would harden the Wilcoxon tests.
- The features are pre-PCA'd, removing interpretability. A real production system would also score raw transaction fields (merchant, country, device).
- We did not test ensemble methods that combine supervised and unsupervised scores — a likely strong baseline.